# Financial Fraud Detection System

## Qwen2.5 QLoRA Fine-Tuning

This notebook performs supervised fine-tuning of
`Qwen/Qwen2.5-1.5B-Instruct` for financial fraud-risk
classification using QLoRA.

The notebook uses the reusable project modules already implemented in
`ml/src/`.

### Training workflow

1. Configure the Colab GPU environment
2. Clone or locate the project repository
3. Install ML dependencies
4. Load the real fraud dataset
5. Validate and preprocess transactions
6. Create leakage-safe train/test partitions
7. Balance the training partition
8. Convert transactions to Qwen conversations
9. Load Qwen in 4-bit NF4
10. Attach LoRA adapters
11. Run supervised fine-tuning
12. Save the trained adapter and tokenizer
13. Save training metadata and training history

The test partition is never used for model optimization.


## 1. GPU Environment Check

QLoRA requires a CUDA-capable GPU for the project's 4-bit
`bitsandbytes` configuration.

In Google Colab select:

**Runtime → Change runtime type → T4 GPU**

before running the training cells.


In [7]:
import os

print("Current directory:", os.getcwd())

for name in [
    "raw_dataset",
    "clean_dataset",
    "train_dataset",
    "test_dataset",
]:
    print(
        name,
        "EXISTS" if name in globals() else "MISSING"
    )

Current directory: /content
raw_dataset MISSING
clean_dataset MISSING
train_dataset MISSING
test_dataset MISSING


In [1]:
import platform

print("Python environment:", platform.python_version())

try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(
            "CUDA version:",
            torch.version.cuda,
        )
    else:
        print(
            "WARNING: CUDA GPU not detected. "
            "Do not start QLoRA training."
        )

except ImportError:
    print(
        "PyTorch is not installed yet. "
        "It will be installed in the dependency step."
    )


Python environment: 3.12.13
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


## 2. Project Setup

The training notebook expects the complete Financial Fraud Detection
System repository.

When running in Colab, clone the GitHub repository after the project has
been pushed to GitHub.

If the repository is already available in the runtime, skip the clone
command and set `PROJECT_ROOT` to the existing repository directory.


In [2]:
from pathlib import Path

COLAB_ROOT = Path("/content")

PROJECT_NAME = "Financial-Fraud-Detection-System"

PROJECT_ROOT = COLAB_ROOT / PROJECT_NAME

print("Expected project root:")
print(PROJECT_ROOT)


Expected project root:
/content/Financial-Fraud-Detection-System


### Repository clone

Before executing this cell in Colab, replace `YOUR_GITHUB_REPOSITORY_URL`
with the repository URL after the project has been pushed to GitHub.

Do not execute the placeholder command unchanged.


In [3]:
# Example for Google Colab:
#
# !git clone YOUR_GITHUB_REPOSITORY_URL /content/Financial-Fraud-Detection-System
#
# After cloning:
#
# %cd /content/Financial-Fraud-Detection-System


## 3. Install Training Dependencies

The project pins the ML stack in `ml/requirements.txt`.

The notebook installs those exact project dependencies so the Colab
training environment matches the repository configuration.


In [4]:
# Run this after the repository has been cloned in Colab:
#
# %cd /content/Financial-Fraud-Detection-System
# !pip install -r ml/requirements.txt


### Runtime restart

If Colab requests a runtime restart after installing PyTorch,
Transformers, TRL, PEFT, or bitsandbytes, restart the runtime and then
continue from the environment verification section below.


In [5]:
# Dependency verification cell.
# Run after installing ml/requirements.txt.

try:
    import accelerate
    import bitsandbytes
    import datasets
    import peft
    import transformers
    import trl

    print("Transformers:", transformers.__version__)
    print("Datasets:", datasets.__version__)
    print("PEFT:", peft.__version__)
    print("TRL:", trl.__version__)
    print("Accelerate:", accelerate.__version__)
    print("bitsandbytes:", bitsandbytes.__version__)

except ImportError as exc:
    print(
        "Dependencies are not fully installed yet:",
        exc,
    )


Dependencies are not fully installed yet: No module named 'bitsandbytes'


## 4. Import Project Pipeline

The notebook does not duplicate the project's ML implementation.

Instead, it imports the reusable loading, preprocessing, splitting,
conversation-formatting, and training modules from `ml/src/`.


In [6]:
from ml.src.data.conversation_format import (
    convert_dataset_to_conversations,
)
from ml.src.data.load_data import (
    load_fraud_dataset,
    select_working_subset,
)
from ml.src.data.preprocess import (
    preprocess_dataset,
    validate_dataset_schema,
)
from ml.src.data.split_data import (
    get_class_counts,
    prepare_balanced_splits,
)
from ml.src.training.train import (
    DEFAULT_MODEL_CONFIG_PATH,
    DEFAULT_TRAINING_CONFIG_PATH,
    get_base_model_id,
    load_yaml_config,
    train_model,
)
from ml.src.utils.constants import (
    DATASET_ID,
    RANDOM_SEED,
    TEST_DATASET_SIZE,
    TRAIN_DATASET_SIZE,
    WORKING_SUBSET_SIZE,
)
from ml.src.utils.seed import set_global_seed

set_global_seed(RANDOM_SEED)

print("Project ML pipeline imported successfully.")


ModuleNotFoundError: No module named 'ml'

## 5. Confirm Frozen Training Configuration

The project configuration is loaded directly from the YAML files committed
to the repository.

This keeps notebook execution consistent with the reusable training engine.


In [ ]:
model_config = load_yaml_config(
    DEFAULT_MODEL_CONFIG_PATH
)

training_config = load_yaml_config(
    DEFAULT_TRAINING_CONFIG_PATH
)

base_model_id = get_base_model_id(
    model_config
)

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Dataset:", DATASET_ID)
print("Base model:", base_model_id)
print("Random seed:", RANDOM_SEED)
print("Working subset:", WORKING_SUBSET_SIZE)
print("Training rows:", TRAIN_DATASET_SIZE)
print("Test rows:", TEST_DATASET_SIZE)

print(
    "Epochs:",
    training_config["training"]["num_train_epochs"],
)

print(
    "Learning rate:",
    training_config["training"]["learning_rate"],
)

print(
    "Batch size:",
    training_config["training"][
        "per_device_train_batch_size"
    ],
)

print(
    "Gradient accumulation:",
    training_config["training"][
        "gradient_accumulation_steps"
    ],
)

print("=" * 70)


## 6. Load the Real Fraud Dataset

The source dataset is:

`CiferAI/Cifer-Fraud-Detection-Dataset-AF`

The project first selects its configured working subset before
preprocessing and model-specific sampling.


In [ ]:
raw_dataset = load_fraud_dataset()

print("Dataset loaded successfully.")
print("Available rows:", len(raw_dataset))
print("Columns:", raw_dataset.column_names)


In [ ]:
working_dataset = select_working_subset(
    raw_dataset,
    subset_size=WORKING_SUBSET_SIZE,
    seed=RANDOM_SEED,
)

print("Working rows:", len(working_dataset))


## 7. Validate and Preprocess Transactions

The preprocessing pipeline:

- verifies required model features
- removes invalid transactions
- normalizes transaction values
- keeps only the features required by the fraud model

`isFlaggedFraud` is intentionally excluded from model inputs because it
could act as a target proxy.


In [ ]:
validate_dataset_schema(
    working_dataset
)

clean_dataset = preprocess_dataset(
    working_dataset
)

print("Preprocessed rows:", len(clean_dataset))
print("Model columns:", clean_dataset.column_names)
print("Class counts:", get_class_counts(clean_dataset))


## 8. Leakage-Safe Train/Test Preparation

The project separates train and test data before balancing.

This is important because balancing before splitting could allow duplicated
or related sampled observations to contaminate model evaluation.

Target sizes:

- Training: 2,000 transactions
- Testing: 500 transactions

Both partitions are balanced independently.


In [ ]:
prepared = prepare_balanced_splits(
    dataset=clean_dataset,
    train_samples_per_class=(
        TRAIN_DATASET_SIZE // 2
    ),
    test_samples_per_class=(
        TEST_DATASET_SIZE // 2
    ),
    seed=RANDOM_SEED,
)

train_dataset = prepared["train"]
test_dataset = prepared["test"]

print("=" * 70)
print("PREPARED DATA")
print("=" * 70)

print(
    "Training:",
    len(train_dataset),
    get_class_counts(train_dataset),
)

print(
    "Testing:",
    len(test_dataset),
    get_class_counts(test_dataset),
)

print("=" * 70)


## 9. Verify Train/Test Separation

The test partition must remain completely independent from the training
partition.

Evaluation will use this test set only after fine-tuning has completed.


In [ ]:
print("Training rows:", len(train_dataset))
print("Testing rows:", len(test_dataset))

print(
    "Training class counts:",
    get_class_counts(train_dataset),
)

print(
    "Testing class counts:",
    get_class_counts(test_dataset),
)

assert len(train_dataset) == TRAIN_DATASET_SIZE
assert len(test_dataset) == TEST_DATASET_SIZE

print("Dataset size checks passed.")


## 10. Preview Qwen Training Conversations

Each transaction becomes a supervised chat conversation.

The user message contains the transaction information and the assistant
target contains the fraud-risk class expected during fine-tuning.


In [ ]:
conversation_preview = (
    convert_dataset_to_conversations(
        train_dataset.select(
            range(min(3, len(train_dataset)))
        )
    )
)

for index in range(
    len(conversation_preview)
):
    print("=" * 70)
    print("EXAMPLE", index + 1)
    print("=" * 70)

    for message in (
        conversation_preview[index]["messages"]
    ):
        print(
            message["role"].upper() + ":"
        )
        print(message["content"])
        print()


## 11. Final GPU Safety Check

The next section performs actual QLoRA fine-tuning.

Do not continue unless CUDA is available.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for the configured "
        "4-bit QLoRA training pipeline."
    )

print("CUDA available:", torch.cuda.is_available())
print("Training GPU:", torch.cuda.get_device_name(0))

gpu_properties = torch.cuda.get_device_properties(0)

print(
    "GPU memory:",
    round(
        gpu_properties.total_memory
        / (1024 ** 3),
        2,
    ),
    "GB",
)

print("GPU safety check passed.")


## 12. QLoRA Fine-Tuning

This is the computationally expensive training cell.

The reusable `train_model()` pipeline will:

1. load Qwen2.5-1.5B-Instruct
2. apply 4-bit NF4 quantization
3. prepare the model for k-bit training
4. attach LoRA adapters
5. render the training conversations
6. create the TRL SFT trainer
7. train for the configured epochs
8. save the LoRA adapter
9. save the tokenizer
10. save reproducibility metadata

The independent test partition is not passed to the trainer.


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(
    "artifacts/fraud-qlora-adapter"
)

print("Adapter output:", OUTPUT_DIR.resolve())
print("Training examples:", len(train_dataset))

trainer = train_model(
    train_dataset=train_dataset,
    output_dir=OUTPUT_DIR,
)

print("QLoRA fine-tuning completed.")


## 13. Training Results

After training, inspect the trainer history.

The loss values generated here will later be used by the evaluation
pipeline and project visualizations.


In [ ]:
training_history = trainer.state.log_history

print("Training history entries:", len(training_history))

for entry in training_history:
    print(entry)


## 14. Save Training History

The adapter directory already contains the model adapter, tokenizer, and
training metadata.

We additionally save the trainer log history as JSON so training curves can
be generated reproducibly during evaluation.


In [ ]:
import json

history_path = (
    OUTPUT_DIR
    / "training_history.json"
)

with history_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        training_history,
        file,
        indent=2,
    )

print(
    "Training history saved:",
    history_path.resolve(),
)


## 15. Verify Saved Adapter

The adapter directory should contain the LoRA adapter configuration and
weights together with tokenizer and project metadata.

These artifacts are much smaller than storing another complete copy of the
base Qwen model.


In [ ]:
print("Saved training artifacts:")

for path in sorted(
    OUTPUT_DIR.rglob("*")
):
    if path.is_file():
        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{path} "
            f"({size_mb:.2f} MB)"
        )


## 16. Preserve the Independent Test Dataset

The test partition will be used in the next project stage to compare:

- base Qwen model performance
- fine-tuned fraud model performance

It must not be used for gradient updates or training decisions.


In [ ]:
print("=" * 70)
print("TRAINING STAGE COMPLETE")
print("=" * 70)

print("Base model:", base_model_id)
print("Training method: QLoRA")
print("Training rows:", len(train_dataset))
print("Held-out test rows:", len(test_dataset))
print("Adapter directory:", OUTPUT_DIR.resolve())

print()
print(
    "The held-out test dataset remains reserved "
    "for model evaluation."
)

print("=" * 70)


## Next Stage

The next notebook will perform model evaluation.

It will compare the original base model with the QLoRA fine-tuned model on
the independent fraud test partition using classification metrics such as:

- accuracy
- precision
- recall
- F1-score
- confusion matrix

The comparison will demonstrate whether domain-specific fine-tuning
improved financial fraud-risk classification.
